# 01. Análise Exploratória, Qualidade e Geografia da Dengue na Paraíba (PB)

**Projeto de Data Science: Dengue no Brasil – Clima, Sazonalidade e Previsão de Casos**  
*Conformidade estrita com as regras de rigor científico (REG-01 a REG-14)*

> **Regra REG-10:** Notebooks destinam-se exclusivamente à narrativa e exploração interativa. Toda a lógica de transformação, validação e modelagem reside nos módulos testados do pacote `src/dengue`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from dengue.config import PROCESSED_DIR, INTERIM_DIR, REPORTS_DIR
from dengue.visualizacao.graficos import (
    calcular_associacao_defasagens_p2,
    clusterizar_perfis_sazonais_p3,
    calcular_autocorrelacao_espacial_moran_p4,
)

# Carrega dados limpos e consolidados gerados pelo pipeline
painel = pd.read_parquet(PROCESSED_DIR / 'painel_semanal_pb.parquet')
funil = pd.read_csv(INTERIM_DIR / 'funil_exclusao_sinan_pb.csv')
print(f"Painel semanal carregado: {len(painel)} registros cobrindo {painel['municipio'].nunique()} municípios da Paraíba.")

## 1. Funil de Exclusão e Qualidade dos Dados (REG-09, REG-12)

Auditamos os registros removidos em cada etapa da limpeza (duplicidades, datas inválidas, municípios fora da PB, idades implausíveis e casos descartados).

In [ ]:
funil[['etapa', 'registros_entrada', 'registros_removidos', 'registros_saida', 'percentual_removido']]

## 2. Pergunta P2: Associação por Defasagem Climática (1 a 12 semanas)

Calculamos as correlações de Spearman entre as variáveis meteorológicas do INMET e a incidência futura de casos por 100 mil habitantes.

In [ ]:
df_p2 = calcular_associacao_defasagens_p2(painel)
df_p2[['defasagem_semanas', 'spearman_temperatura', 'spearman_chuva', 'spearman_umidade']]

## 3. Perguntas P3 e P4: Clusters Sazonais e Autocorrelação Espacial de Moran

- **P3:** Identifica clusters de comportamento sazonal (litoral vs interior semiárido).
- **P4:** Avalia a autocorrelação espacial via Índice Global de Moran com matriz $k$-NN geodésica.

In [ ]:
df_p3 = clusterizar_perfis_sazonais_p3(painel)
print("Exemplo de clusters sazonais por mesorregião:")
print(df_p3.groupby(['mesorregiao', 'cluster_sazonal'])['municipio'].count())

moran = calcular_autocorrelacao_espacial_moran_p4(painel)
print(f"Índice I de Moran: {moran['moran_i']} | p-valor: {moran['p_valor_permutacao']}")